[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Analytics_Pipeline.ipynb)

# Analytics Pipeline — Bronze to Silver to Gold

**The complete Data Engineering pipeline in one notebook.**

Loads raw call center data → cleans it → builds a star schema → runs analytical queries.

No cloud. No orchestration. Just Python + Pandas. This is where every pipeline starts.

In [ ]:
import pandas as pd
import numpy as np
import os

# On Colab: clone the repo to get the data
if not os.path.exists("data") and not os.path.exists("../../data"):
    print("Downloading data from GitHub...")
    !git clone --depth 1 https://github.com/sunilmogadati/systems-in-production.git /tmp/repo 2>/dev/null
    DATA_DIR = "/tmp/repo/data"
elif os.path.exists("../../data/calls.json"):
    DATA_DIR = "../../data"
elif os.path.exists("data/calls.json"):
    DATA_DIR = "data"
else:
    DATA_DIR = os.path.expanduser("~/Downloads/systems-in-production/data")

print(f"Data directory: {DATA_DIR}")
print(f"Files: {os.listdir(DATA_DIR)}")

---

## Stage 1: BRONZE — Load Raw Data

Bronze = raw, untouched. Every record preserved. No cleaning.

**WHY:** This is the source of truth. If anything goes wrong downstream, we can always come back here and reprocess from scratch. In production, this would be files landing in GCS (Google Cloud Storage) or S3.

In [ ]:
# Load all raw data sources
# calls.json is newline-delimited JSON (each line = one call record)
# The rest are CSV files

calls_raw = pd.read_json(os.path.join(DATA_DIR, "calls.json"), lines=True)
campaigns_raw = pd.read_csv(os.path.join(DATA_DIR, "campaigns.csv"))
products_raw = pd.read_csv(os.path.join(DATA_DIR, "products.csv"))
orders_raw = pd.read_csv(os.path.join(DATA_DIR, "orders.csv"))

print(f"calls:     {len(calls_raw):,} records, columns: {list(calls_raw.columns)}")
print(f"campaigns: {len(campaigns_raw):,} records")
print(f"products:  {len(products_raw):,} records")
print(f"orders:    {len(orders_raw):,} records")

In [ ]:
# ALWAYS look at the data before doing anything
# What does a raw call record look like?
calls_raw.head(3)

In [ ]:
# Check data types and nulls
calls_raw.info()

In [ ]:
# What do campaigns look like?
campaigns_raw

In [ ]:
# What do orders look like?
orders_raw.head()

---

## Stage 2: SILVER — Clean the Data

Silver = cleaned but **complete**. Duplicates removed, types fixed, timestamps standardized.

**Critical rule: Missing values are FLAGGED, not dropped.**

WHY: Analytics needs every record for accurate counts. "How many total calls?" must be exact — even if some calls have missing fields. The ML pipeline (separate notebook) drops records with missing features. Same Silver layer, different Gold consumers.

In [ ]:
# Start with a copy — never modify Bronze
calls = calls_raw.copy()

# --- Flatten nested customer data ---
# The JSON has a nested "customer" object. Flatten it for SQL-like queries.
if "customer" in calls.columns:
    customer_df = pd.json_normalize(calls["customer"])
    customer_df.columns = ["customer_" + c.replace(".", "_") for c in customer_df.columns]
    calls = pd.concat([calls.drop(columns=["customer"]), customer_df], axis=1)
    print(f"Flattened {len(customer_df.columns)} customer fields")

print(f"Columns after flattening: {list(calls.columns)}")

In [ ]:
# --- Deduplicate ---
# WHY: Source systems sometimes send the same call twice.
# Duplicate calls inflate counts and revenue.
before = len(calls)
calls = calls.drop_duplicates(subset=["call_id"], keep="first")
after = len(calls)
print(f"Duplicates removed: {before - after}")
print(f"Records: {before} → {after}")

In [ ]:
# --- Fix data types ---
# Dates arrive as strings. Pandas needs them as datetime for filtering/grouping.
calls["date"] = pd.to_datetime(calls["date"], errors="coerce")
calls["start_time"] = pd.to_datetime(calls["start_time"], errors="coerce", utc=True)
calls["end_time"] = pd.to_datetime(calls["end_time"], errors="coerce", utc=True)
calls["dnis"] = calls["dnis"].astype(str)

# Standardize text
calls["disposition"] = calls["disposition"].str.strip().str.lower()
calls["channel"] = calls["channel"].str.strip().str.upper()

print("Data types fixed.")
calls.dtypes

In [ ]:
# --- Flag missing values (do NOT drop them) ---
# WHY: The ML pipeline drops these. The analytics pipeline keeps them.
# Both work from Silver. The difference is in the Gold layer.

for col in ["duration", "disposition", "start_time"]:
    if col in calls.columns:
        flag_col = f"has_missing_{col}"
        calls[flag_col] = calls[col].isna()
        missing = calls[flag_col].sum()
        total = len(calls)
        print(f"  {col}: {missing}/{total} missing ({100*missing/total:.1f}%) — flagged, not dropped")

print(f"\nSilver calls: {len(calls):,} records (all preserved)")

---

## Stage 3: GOLD (Analytics) — Build the Star Schema

Gold = analytical-ready. Dimension tables (context) + fact table (measurements).

**WHY:** The star schema makes "average duration by campaign by day of week" a simple GROUP BY instead of a complex multi-join with date parsing.

In [ ]:
# --- Dimension: dim_date ---
# Pre-computed calendar. Analysts filter by day name, weekend, month
# without parsing timestamps at query time.

unique_dates = calls["date"].dropna().dt.date.unique()
dim_date = pd.DataFrame({"full_date": sorted(unique_dates)})
dim_date["date_key"] = range(1, len(dim_date) + 1)
dim_date["full_date"] = pd.to_datetime(dim_date["full_date"])
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["is_weekend"] = dim_date["full_date"].dt.dayofweek >= 5

print(f"dim_date: {len(dim_date)} rows")
dim_date.head()

In [ ]:
# --- Dimension: dim_campaign ---
# Campaign attributes in one place. Surrogate key for fast joins.

campaigns = campaigns_raw.copy()
campaigns["dnis"] = campaigns["dnis"].astype(str)
campaigns["campaign_key"] = range(1, len(campaigns) + 1)
dim_campaign = campaigns

print(f"dim_campaign: {len(dim_campaign)} rows")
dim_campaign

In [ ]:
# --- Fact table: fact_calls ---
# One row per call. Surrogate keys point to dimensions.

fact_calls = calls.copy()

# Join campaign key
fact_calls = fact_calls.merge(
    dim_campaign[["dnis", "campaign_key", "campaign_name", "channel"]].rename(
        columns={"channel": "campaign_channel"}),
    on="dnis", how="left"
)

# Join date key
fact_calls["date_str"] = fact_calls["date"].dt.date.astype(str)
dim_date["date_str"] = dim_date["full_date"].dt.date.astype(str)
fact_calls = fact_calls.merge(dim_date[["date_str", "date_key"]], on="date_str", how="left")
fact_calls.drop(columns=["date_str"], inplace=True)
dim_date.drop(columns=["date_str"], inplace=True)

# Extract hour for time-of-day analysis
if "start_time" in fact_calls.columns:
    fact_calls["hour_of_day"] = fact_calls["start_time"].dt.hour

print(f"fact_calls: {len(fact_calls):,} rows")
print(f"Columns: {list(fact_calls.columns)}")

---

## Stage 4: Analytical Queries — The Payoff

These are the questions the VP asks at 3 AM. On the star schema, each is a simple GROUP BY.

In [ ]:
# Query 1: Average call duration by campaign
# This is the #1 report in any call center: which campaigns are performing?

q1 = fact_calls.groupby(["campaign_name", "campaign_channel"]).agg(
    avg_duration=("duration", "mean"),
    total_calls=("call_id", "count")
).round(1).sort_values("avg_duration", ascending=False)

print("Campaign Performance — Average Call Duration")
q1

In [ ]:
# Query 2: Calls by day of week
# Staffing decisions — when do we need more agents?

q2 = fact_calls.merge(dim_date[["date_key", "day_name", "is_weekend"]], on="date_key", how="left")
result2 = q2.groupby(["day_name", "is_weekend"]).agg(
    total_calls=("call_id", "count")
).sort_values("total_calls", ascending=False)

print("Calls by Day of Week")
result2

In [ ]:
# Query 3: Disposition (outcome) distribution
# Why are calls ending the way they do?

print("Call Disposition Distribution")
fact_calls["disposition"].value_counts()

In [ ]:
# Query 4: VA vs Live channel comparison
# Is the Virtual Agent performing as well as human agents?

q4 = fact_calls.groupby("campaign_channel").agg(
    total_calls=("call_id", "count"),
    avg_duration=("duration", "mean")
).round(1)

print("Channel Comparison: VA vs Live")
q4

In [ ]:
# Query 5: Data Quality Report
# How many records have missing values? This is what the operations team needs to fix.

print("Data Quality Report")
print("=" * 50)
missing_cols = [c for c in fact_calls.columns if c.startswith("has_missing_")]
for col in missing_cols:
    field = col.replace("has_missing_", "")
    missing = fact_calls[col].sum()
    total = len(fact_calls)
    pct = 100.0 * missing / total
    status = "OK" if missing == 0 else "INVESTIGATE"
    print(f"  {field:20s}: {missing:3d}/{total} missing ({pct:.1f}%) [{status}]")

---

## Summary

What we built:

In [ ]:
print("ANALYTICS PIPELINE COMPLETE")
print("=" * 50)
print(f"Bronze: {len(calls_raw):,} raw records")
print(f"Silver: {len(calls):,} cleaned records (all preserved)")
print(f"Gold:   {len(fact_calls):,} fact rows | {len(dim_date)} dates | {len(dim_campaign)} campaigns")
print()
print("Pipeline: Bronze (raw) → Silver (clean, flagged) → Gold (star schema) → Queries")
print()
print("This runs on 500 calls in 2 seconds.")
print("At 50 million calls, Pandas runs out of memory.")
print("That is why the next step is GCS + BigQuery + Cloud Composer.")